In [2]:
# snmf_env
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Rectangle
from matplotlib.colors import LogNorm
from matplotlib import cm
import os
import pickle as pkl
from pathlib import Path


In [ ]:
#
#! TEMP - i

# Load Data
# combine X_train and X_test (add _[test/train])

N_train = 100
path_trainfold = os.path.join(project_root, "data/processed/bootstrapped_sameSplit/N_{}".format(N_train)) 
N_test = 1000
path_testfold = os.path.join(project_root, "data/processed/bootstrapped_sameSplit/N_{}".format(N_test)) 

train_1 = pd.read_pickle(path_trainfold + "/train_1.pkl")
train_2 = pd.read_pickle(path_trainfold + "/train_2.pkl")
train_3 = pd.read_pickle(path_trainfold + "/train_3.pkl")
test = pd.read_pickle(path_testfold + "/test.pkl")
# test = pd.read_pickle(test_dir + "/test.pkl")
features = train_1.columns[:96]

train_1.index = 'train1_'+ train_1['Gene_KO'] + train_1['boot'].replace({False: '_real', True: '_boot'})
train_2.index = 'train2_'+ train_2['Gene_KO'] + train_2['boot'].replace({False: '_real', True: '_boot'})
train_3.index = 'train3_'+ train_3['Gene_KO'] + train_3['boot'].replace({False: '_real', True: '_boot'})
test.index = 'test_'+ test['Gene_KO'] + test['boot'].replace({False: '_real', True: '_boot'})

X_train1 = train_1.loc[:,features]
y_train1 = train_1.loc[:, "label"]
X_train2 = train_2.loc[:,features]
y_train2 = train_2.loc[:, "label"]
X_train3 = train_3.loc[:,features]
y_train3 = train_3.loc[:, "label"]
X_test = test.loc[:,features]
y_test = test.loc[:, "label"]

# Concatenate the DataFrames along the columns (axis=1)
df_X_all = pd.concat([X_train1.T, X_train2.T, X_train3.T, X_test.T], axis=1)

df_X_all = df_X_all * 1000
df_X_all.index.name = 'Mutation Types'

path_X_all = os.path.join(project_root,"data/processed/bootstrapped_sameSplit_sorted/X_allx1000_new.text")
df_X_all.to_csv(path_X_all, sep="\t")



In [ ]:
import os
import pandas as pd

# --- Configuration ---
project_root = "/Users/sande/Documents/GitHub/SNMF"
N_train = 100
N_test = 1000
path_trainfold = os.path.join(project_root, f"data/processed/bootstrapped_sameSplit/N_{N_train}")
path_testfold = os.path.join(project_root, f"data/processed/bootstrapped_sameSplit/N_{N_test}")
output_real_dir = os.path.join(project_root, "data/processed/real")
os.makedirs(output_real_dir, exist_ok=True)

# --- Load data ---
train_1 = pd.read_pickle(os.path.join(path_trainfold, "train_1.pkl"))
train_2 = pd.read_pickle(os.path.join(path_trainfold, "train_2.pkl"))
train_3 = pd.read_pickle(os.path.join(path_trainfold, "train_3.pkl"))
test = pd.read_pickle(os.path.join(path_testfold, "test.pkl"))

# --- Add source label ---
train_1["source"] = "train1"
train_2["source"] = "train2"
train_3["source"] = "train3"
test["source"] = "test"

# --- Combine all data ---
df_all = pd.concat([train_1, train_2, train_3, test], axis=0)

# --- Filter to real samples only ---
df_real = df_all[df_all["boot"] == False].copy()

# --- Add split column (train/test) ---
df_real["split"] = df_real["source"].apply(lambda s: "train" if "train" in s else "test")

# --- Create Gene_KO_paired column with unique identifiers ---
df_real["Gene_KO_paired"] = (
    df_real.groupby("Gene_KO").cumcount().astype(str)
)
df_real["Gene_KO_paired"] = df_real["Gene_KO"] + "_" + df_real["Gene_KO_paired"]

# --- Set index for clarity ---
df_real.index = df_real["source"] + "_" + df_real["Gene_KO_paired"] + "_real"

# --- Save full DataFrame ---
output_path = os.path.join(output_real_dir, "zou_all.pkl")
df_real.to_pickle(output_path)

print(f"✅ Saved full real data with unique KO identifiers to: {output_path}")


# Split in multiple folds

5  random train/test splits 
* 1 sample from each gene KO in test (2 for control)
* remainder in train

In [3]:
project_root = Path.cwd()
while not (project_root / "data").exists() and project_root != project_root.parent:
    project_root = project_root.parent
project_root


PosixPath('/Users/sande/Projects/SNMF')

In [4]:
data_path = project_root / "data" / "raw" / "zou2021" / "zou_96SBS_filtered.pkl"
print("Loading:", data_path)


df = pd.read_pickle(data_path)
print("Shape:", df.shape)
print(df.columns)
print(df.index)
df

Loading: /Users/sande/Projects/SNMF/data/raw/zou2021/zou_96SBS_filtered.pkl
Shape: (46, 100)
Index(['A[C>A]A', 'A[C>A]C', 'A[C>A]G', 'A[C>A]T', 'A[C>G]A', 'A[C>G]C',
       'A[C>G]G', 'A[C>G]T', 'A[C>T]A', 'A[C>T]C', 'A[C>T]G', 'A[C>T]T',
       'A[T>A]A', 'A[T>A]C', 'A[T>A]G', 'A[T>A]T', 'A[T>C]A', 'A[T>C]C',
       'A[T>C]G', 'A[T>C]T', 'A[T>G]A', 'A[T>G]C', 'A[T>G]G', 'A[T>G]T',
       'C[C>A]A', 'C[C>A]C', 'C[C>A]G', 'C[C>A]T', 'C[C>G]A', 'C[C>G]C',
       'C[C>G]G', 'C[C>G]T', 'C[C>T]A', 'C[C>T]C', 'C[C>T]G', 'C[C>T]T',
       'C[T>A]A', 'C[T>A]C', 'C[T>A]G', 'C[T>A]T', 'C[T>C]A', 'C[T>C]C',
       'C[T>C]G', 'C[T>C]T', 'C[T>G]A', 'C[T>G]C', 'C[T>G]G', 'C[T>G]T',
       'G[C>A]A', 'G[C>A]C', 'G[C>A]G', 'G[C>A]T', 'G[C>G]A', 'G[C>G]C',
       'G[C>G]G', 'G[C>G]T', 'G[C>T]A', 'G[C>T]C', 'G[C>T]G', 'G[C>T]T',
       'G[T>A]A', 'G[T>A]C', 'G[T>A]G', 'G[T>A]T', 'G[T>C]A', 'G[T>C]C',
       'G[T>C]G', 'G[T>C]T', 'G[T>G]A', 'G[T>G]C', 'G[T>G]G', 'G[T>G]T',
       'T[C>A]A', 'T[C>A]C', 'T

,A[C>A]A,A[C>A]C,A[C>A]G,A[C>A]T,A[C>G]A,A[C>G]C,A[C>G]G,A[C>G]T,A[C>T]A,A[C>T]C,...,T[T>C]G,T[T>C]T,T[T>G]A,T[T>G]C,T[T>G]G,T[T>G]T,Gene_KO,Count,Protein_KO,Subpathway_KO
EXO1_71_s2,0.028554,0.013366,0.006683,0.013973,0.009721,0.008505,0.009113,0.006075,0.012151,0.011543,...,0.010936,0.010936,0.007898,0.003038,0.007290,0.007898,EXO1,1646,Exonuclease 1,HR and HR regulation
EXO1_71_s3,0.023744,0.017352,0.008219,0.019178,0.009132,0.013699,0.006393,0.012785,0.017352,0.006393,...,0.008219,0.008219,0.005479,0.001826,0.003653,0.012785,EXO1,1095,Exonuclease 1,HR and HR regulation
EXO1_71_s4,0.016562,0.014984,0.004732,0.014196,0.010252,0.005521,0.007098,0.006309,0.019716,0.010252,...,0.007098,0.020505,0.003155,0.003943,0.004732,0.007098,EXO1,1268,Exonuclease 1,HR and HR regulation
OGG1_106_s1,0.115299,0.004435,0.008869,0.022173,0.000000,0.000000,0.000000,0.000000,0.006652,0.006652,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.002217,OGG1,451,8-Oxoguanine glycosylase,BER
OGG1_106_s2,0.131336,0.009217,0.004608,0.027650,0.002304,0.000000,0.002304,0.002304,0.013825,0.006912,...,0.004608,0.000000,0.000000,0.000000,0.000000,0.002304,OGG1,434,8-Oxoguanine glycosylase,BER
OGG1_25_s1,0.136490,0.006964,0.001393,0.018106,0.001393,0.000000,0.000000,0.000000,0.009749,0.001393,...,0.002786,0.002786,0.002786,0.000000,0.001393,0.001393,OGG1,718,8-Oxoguanine glycosylase,BER
OGG1_25_s2,0.133949,0.008083,0.002309,0.025404,0.003464,0.002309,0.000000,0.003464,0.008083,0.002309,...,0.002309,0.005774,0.002309,0.000000,0.002309,0.004619,OGG1,866,8-Oxoguanine glycosylase,BER
RNF168_116_s1,0.035183,0.018945,0.006766,0.013532,0.013532,0.006766,0.012179,0.009472,0.024357,0.005413,...,0.002706,0.020298,0.002706,0.005413,0.008119,0.014885,RNF168,739,Ring finger protein 168,Checkpoint/DSB repair
RNF168_116_s2,0.027097,0.010323,0.005161,0.012903,0.009032,0.001290,0.003871,0.018065,0.028387,0.009032,...,0.009032,0.020645,0.003871,0.005161,0.003871,0.012903,RNF168,775,Ring finger protein 168,Checkpoint/DSB repair
RNF168_14_s1,0.025735,0.003676,0.003676,0.003676,0.003676,0.003676,0.018382,0.018382,0.025735,0.014706,...,0.000000,0.007353,0.003676,0.000000,0.000000,0.014706,RNF168,272,Ring finger protein 168,Checkpoint/DSB repair


# Split Train in 3 sub-folds 

In [5]:
from pathlib import Path
import pandas as pd
import numpy as np
import sys, os, importlib

# --- Add project src to path for import ---
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../..")))
from src.processing import bootstrapping as boot
importlib.reload(boot)

# --- Constants ---
N_train = 100
N_test = 1000  
ds = 100
n_splits = 10
n_folds = 3
fixed_class_order = ["Control", "MMR", "HR", "BER"]

# --- Get project root dynamically ---
project_root = Path.cwd()
while not (project_root / "data").exists() and project_root != project_root.parent:
    project_root = project_root.parent

# --- Load dataset ---
data_path = project_root / "data" / "raw" / "zou2021" / "zou_96SBS_filtered.pkl"
print("Loading:", data_path)
df = pd.read_pickle(data_path)
print("Shape:", df.shape)

# --- Map Gene_KO → Pathway class ---
gene_to_pathway = {
    "ATP2B4": "Control",
    "MSH2": "MMR", "MSH6": "MMR", "MLH1": "MMR", "PMS1": "MMR", "PMS2": "MMR",
    "EXO1": "HR", "RNF168": "HR",
    "UNG": "BER", "OGG1": "BER",
}
df["label_name"] = df["Gene_KO"].map(gene_to_pathway)
if df["label_name"].isnull().any():
    missing = df[df["label_name"].isnull()]["Gene_KO"].unique()
    raise ValueError(f"Unmapped genes: {missing}")
label_map = {name: i for i, name in enumerate(fixed_class_order)}
df["label"] = df["label_name"].map(label_map)
df["Gene_KO_paired"] = df["Gene_KO"].astype(str)

print("Label distribution:")
print(df["label_name"].value_counts())

# --- Output folder ---
out_root = project_root / "data" / "processed" / "zou2021" / "splits_new"
out_root.mkdir(parents=True, exist_ok=True)

rng_global = np.random.default_rng(42)

# --- Helper ---
def save_bootstrapped_split(Xboot, suffix, outdir, prefix="D"):
    path_X = os.path.join(outdir, f"Xboot{prefix}_{suffix}.txt")
    path_Y = os.path.join(outdir, f"Yboot{prefix}_{suffix}.txt")
    mut_cols = Xboot.columns[:96]
    Xboot[mut_cols].T.to_csv(path_X, sep="\t")
    y_onehot = pd.get_dummies(Xboot["label_name"]).T
    y_onehot = y_onehot.reindex(fixed_class_order, fill_value=0)
    y_onehot.columns = Xboot.index
    y_onehot.to_csv(path_Y, sep="\t")

# ---------------------------------------------------------------------
# Generate new splits
# ---------------------------------------------------------------------
for split_id in range(n_splits):
    rng = np.random.default_rng(42 + split_id)  # unique seed per split

    # --- Gene-wise balanced test selection ---
    te_idx, tr_idx = [], []
    for gene, subdf in df.groupby("Gene_KO"):
        n_remove = 2 if gene == "ATP2B4" else 1  # remove 2 from ATP2B4, 1 otherwise
        n_remove = min(n_remove, len(subdf) - 1)  # safeguard
        test_samples = rng.choice(subdf.index, size=n_remove, replace=False)
        train_samples = subdf.index.difference(test_samples)
        te_idx.extend(test_samples)
        tr_idx.extend(train_samples)

    df_test = df.loc[te_idx]
    df_train = df.loc[tr_idx]

    split_dir = out_root / f"split_{split_id:02d}"
    split_dir.mkdir(parents=True, exist_ok=True)
    df_train.to_pickle(split_dir / "train.pkl")
    df_test.to_pickle(split_dir / "test.pkl")

    print(f"\nSplit {split_id:02d}: train={len(df_train)} | test={len(df_test)}")

    # -----------------------------------------------------------------
    # Split train into 3 folds (gene-balanced + randomized remainder)
    # -----------------------------------------------------------------
    folds = {i: [] for i in range(n_folds)}
    for gene, subdf in df_train.groupby("Gene_KO"):
        samples = subdf.index.to_list()
        rng.shuffle(samples)  # randomize order per gene

        base, rem = divmod(len(samples), n_folds)
        fold_sizes = [base] * n_folds
        # randomly assign remainder to folds
        if rem > 0:
            for i in rng.choice(n_folds, size=rem, replace=False):
                fold_sizes[i] += 1

        start = 0
        for f, size in enumerate(fold_sizes):
            end = start + size
            folds[f].extend(samples[start:end])
            start = end

    subfolds_dir = split_dir / "sub_folds"
    subfolds_dir.mkdir(exist_ok=True)

    XbootM_all, XbootD_all = [], []

    # -----------------------------------------------------------------
    # Bootstrap each fold and combine
    # -----------------------------------------------------------------
    for i in range(n_folds):
        fold_idx = folds[i]
        fold_df = df_train.loc[fold_idx]
        fold_dir = subfolds_dir / f"fold_{i+1}"
        fold_dir.mkdir(parents=True, exist_ok=True)

        print(f"→ Bootstrapping split {split_id:02d}, fold {i+1} ({len(fold_df)} samples)")

        XbootM_train, XbootD_train = boot.bootstrap_dirichlet(
            fold_df, L=N_train, dirichlet_strength=ds, suffix=f"train_fold{i+1}", class_col="label"
        )

        save_bootstrapped_split(XbootM_train, f"train_fold{i+1}", fold_dir, prefix="M")
        save_bootstrapped_split(XbootD_train, f"train_fold{i+1}", fold_dir, prefix="D")

        XbootM_all.append(XbootM_train)
        XbootD_all.append(XbootD_train)

    # -----------------------------------------------------------------
    # Combine folds into train_all
    # -----------------------------------------------------------------
    XbootM_all = pd.concat(XbootM_all, axis=0)
    XbootD_all = pd.concat(XbootD_all, axis=0)

    save_bootstrapped_split(XbootM_all, "train_all", split_dir, prefix="M")
    save_bootstrapped_split(XbootD_all, "train_all", split_dir, prefix="D")

    # -----------------------------------------------------------------
    # Bootstrap the small test set (L=4000)
    # -----------------------------------------------------------------
    print(f"→ Bootstrapping test (11 samples → 4000)...")
    XbootM_test, XbootD_test = boot.bootstrap_dirichlet(
        df_test, L=N_test, dirichlet_strength=ds, suffix="test", class_col="label"
    )

    save_bootstrapped_split(XbootM_test, "test", split_dir, prefix="M")
    save_bootstrapped_split(XbootD_test, "test", split_dir, prefix="D")

    print(f"✅ Saved split_{split_id:02d}: train={len(df_train)} | test={len(df_test)}")

print(f"\n✅ Done. Bootstrapped splits saved under:\n{out_root}")


Loading: /Users/sande/Projects/SNMF/data/raw/zou2021/zou_96SBS_filtered.pkl
Shape: (46, 100)
Label distribution:
label_name
MMR        23
BER         8
Control     8
HR          7
Name: count, dtype: int64

Split 00: train=35 | test=11
→ Bootstrapping split 00, fold 1 (12 samples)
→ Bootstrapping split 00, fold 2 (11 samples)
→ Bootstrapping split 00, fold 3 (12 samples)
→ Bootstrapping test (11 samples → 4000)...
✅ Saved split_00: train=35 | test=11

Split 01: train=35 | test=11
→ Bootstrapping split 01, fold 1 (11 samples)
→ Bootstrapping split 01, fold 2 (11 samples)
→ Bootstrapping split 01, fold 3 (13 samples)
→ Bootstrapping test (11 samples → 4000)...
✅ Saved split_01: train=35 | test=11

Split 02: train=35 | test=11
→ Bootstrapping split 02, fold 1 (11 samples)
→ Bootstrapping split 02, fold 2 (12 samples)
→ Bootstrapping split 02, fold 3 (12 samples)
→ Bootstrapping test (11 samples → 4000)...
✅ Saved split_02: train=35 | test=11

Split 03: train=35 | test=11
→ Bootstrapping s

In [41]:
import pandas as pd
from pathlib import Path

# --- Define which split to inspect ---
split_id = 0  # change to e.g. 1, 2, etc.
split_dir = Path("/Users/sande/Projects/SNMF/data/processed/zou2021/splits_new") / f"split_{split_id:02d}"

# --- Load multinomial bootstrapped train data ---
X_path = split_dir / "XbootM_train_all.txt"
Y_path = split_dir / "YbootM_train_all.txt"

print(f"Loading from: {split_dir}")
X = pd.read_csv(X_path, sep="\t", index_col=0)
Y = pd.read_csv(Y_path, sep="\t", index_col=0)

print("\n✅ Shapes:")
print(f"X: {X.shape}  (mutational profiles: 96 features x {X.shape[1]} samples)")
print(f"Y: {Y.shape}  (pathway one-hot: {Y.shape[0]} classes x {Y.shape[1]} samples)")

# --- Sanity check: matching sample order ---
if X.columns.equals(Y.columns):
    print("✅ Sample alignment: MATCH")
else:
    print("⚠️ Sample mismatch between X and Y")

# --- Quick preview ---
print("\nClasses (rows in Y):", list(Y.index))
print("\nExample sample IDs:", list(X.columns[:5]))
print("\nExample X snippet:")
print(X.iloc[:5, :5])

print("\nExample Y snippet:")
print(Y.iloc[:, :5])


Loading from: /Users/sande/Projects/SNMF/data/processed/zou2021/splits_new/split_00

✅ Shapes:
X: (96, 1200)  (mutational profiles: 96 features x 1200 samples)
Y: (4, 1200)  (pathway one-hot: 4 classes x 1200 samples)
✅ Sample alignment: MATCH

Classes (rows in Y): ['Control', 'MMR', 'HR', 'BER']

Example sample IDs: ['ATP2B4_real_train_fold1', 'ATP2B4_real_train_fold1.1', 'MLH1_real_train_fold1', 'MSH2_real_train_fold1', 'MSH6_real_train_fold1']

Example X snippet:
         ATP2B4_real_train_fold1  ATP2B4_real_train_fold1.1  \
A[C>A]A                 0.087838                   0.076596   
A[C>A]C                 0.006757                   0.004255   
A[C>A]G                 0.000000                   0.000000   
A[C>A]T                 0.010135                   0.025532   
A[C>G]A                 0.006757                   0.000000   

         MLH1_real_train_fold1  MSH2_real_train_fold1  MSH6_real_train_fold1  
A[C>A]A               0.005546               0.004322               0.0

In [38]:
import pandas as pd
from pathlib import Path

# --- Locate your split ---
project_root = Path.cwd()
while not (project_root / "data").exists() and project_root != project_root.parent:
    project_root = project_root.parent

split_path = project_root / "data" / "processed" / "zou2021" / "splits_new" / "split_01" / "train.pkl"

# --- Load ---
df_train = pd.read_pickle(split_path)
print(f"Shape: {df_train.shape}")
print("\nLabel distribution (label_name):")
print(df_train["label_name"].value_counts())
df_train


Shape: (35, 103)

Label distribution (label_name):
label_name
MMR        18
Control     6
BER         6
HR          5
Name: count, dtype: int64


,A[C>A]A,A[C>A]C,A[C>A]G,A[C>A]T,A[C>G]A,A[C>G]C,A[C>G]G,A[C>G]T,A[C>T]A,A[C>T]C,...,T[T>G]C,T[T>G]G,T[T>G]T,Gene_KO,Count,Protein_KO,Subpathway_KO,label_name,label,Gene_KO_paired
ATP2B4_2_s3,0.069620,0.012658,0.000000,0.006329,0.000000,0.006329,0.000000,0.000000,0.037975,0.012658,...,0.000000,0.000000,0.000000,ATP2B4,158,Plasma membrane calcium-transporting ATPase 4,Control,Control,0,ATP2B4
ATP2B4_2_s4,0.048673,0.004425,0.000000,0.013274,0.000000,0.000000,0.000000,0.000000,0.035398,0.000000,...,0.000000,0.000000,0.004425,ATP2B4,226,Plasma membrane calcium-transporting ATPase 4,Control,Control,0,ATP2B4
ATP2B4_2_s5,0.030075,0.000000,0.007519,0.007519,0.000000,0.000000,0.000000,0.000000,0.015038,0.022556,...,0.000000,0.000000,0.015038,ATP2B4,133,Plasma membrane calcium-transporting ATPase 4,Control,Control,0,ATP2B4
ATP2B4_5_s4,0.068493,0.000000,0.006849,0.047945,0.000000,0.000000,0.000000,0.006849,0.041096,0.006849,...,0.006849,0.000000,0.006849,ATP2B4,146,Plasma membrane calcium-transporting ATPase 4,Control,Control,0,ATP2B4
ATP2B4_5_s5,0.076596,0.004255,0.000000,0.025532,0.000000,0.008511,0.000000,0.004255,0.029787,0.012766,...,0.004255,0.004255,0.004255,ATP2B4,235,Plasma membrane calcium-transporting ATPase 4,Control,Control,0,ATP2B4
ATP2B4_5_s8,0.087838,0.006757,0.000000,0.010135,0.006757,0.003378,0.000000,0.003378,0.023649,0.020270,...,0.000000,0.000000,0.003378,ATP2B4,296,Plasma membrane calcium-transporting ATPase 4,Control,Control,0,ATP2B4
EXO1_71_s2,0.028554,0.013366,0.006683,0.013973,0.009721,0.008505,0.009113,0.006075,0.012151,0.011543,...,0.003038,0.007290,0.007898,EXO1,1646,Exonuclease 1,HR and HR regulation,HR,2,EXO1
EXO1_71_s3,0.023744,0.017352,0.008219,0.019178,0.009132,0.013699,0.006393,0.012785,0.017352,0.006393,...,0.001826,0.003653,0.012785,EXO1,1095,Exonuclease 1,HR and HR regulation,HR,2,EXO1
MLH1_172_s1,0.009259,0.003899,0.000975,0.008285,0.003899,0.000000,0.000487,0.002437,0.062378,0.014620,...,0.001949,0.001949,0.000975,MLH1,2052,MutL homolog 1,MMR,MMR,1,MLH1
MLH1_172_s2,0.004132,0.002066,0.001033,0.008264,0.002066,0.000517,0.000000,0.003099,0.059917,0.022211,...,0.000517,0.003099,0.001550,MLH1,1936,MutL homolog 1,MMR,MMR,1,MLH1


In [30]:
import pandas as pd
from pathlib import Path

# --- Locate your project ---
project_root = Path.cwd()
while not (project_root / "data").exists() and project_root != project_root.parent:
    project_root = project_root.parent

# --- Load full dataset ---
df_full = pd.read_pickle(project_root / "data" / "raw" / "zou2021" / "zou_96SBS_filtered.pkl")

# --- Load train split ---
split_dir = project_root / "data" / "processed" / "zou2021" / "splits_new" / "split_00"
df_train = pd.read_pickle(split_dir / "train.pkl")

# --- Count per gene ---
print("\n🧬 Counts per Gene_KO (full dataset):")
print(df_full["Gene_KO"].value_counts().sort_index())

print("\n📦 Counts per Gene_KO (split_00 TRAIN):")
print(df_train["Gene_KO"].value_counts().sort_index())

print(f"\nTotal in full dataset: {len(df_full)}  |  Train: {len(df_train)}")



🧬 Counts per Gene_KO (full dataset):
Gene_KO
ATP2B4    8
EXO1      3
MLH1      4
MSH2      3
MSH6      8
OGG1      4
PMS1      4
PMS2      4
RNF168    4
UNG       4
Name: count, dtype: int64

📦 Counts per Gene_KO (split_00 TRAIN):
Gene_KO
ATP2B4    7
EXO1      3
MLH1      3
MSH2      1
MSH6      7
OGG1      2
PMS1      2
PMS2      3
RNF168    3
UNG       4
Name: count, dtype: int64

Total in full dataset: 46  |  Train: 35


# Add Original split as split_10

In [43]:
N_train = 100
N_test = 1000
path_trainfold = os.path.join(project_root, f"data/processed/bootstrapped_sameSplit/N_{N_train}")
path_testfold = os.path.join(project_root, f"data/processed/bootstrapped_sameSplit/N_{N_test}")
output_real_dir = os.path.join(project_root, "data/processed/real")
os.makedirs(output_real_dir, exist_ok=True)

# --- Load data ---
train_1 = pd.read_pickle(os.path.join(path_trainfold, "train_1.pkl"))
train_2 = pd.read_pickle(os.path.join(path_trainfold, "train_2.pkl"))
train_3 = pd.read_pickle(os.path.join(path_trainfold, "train_3.pkl"))
test = pd.read_pickle(os.path.join(path_testfold, "test.pkl"))


In [45]:
train_1

,A[C>A]A,A[C>A]C,A[C>A]G,A[C>A]T,A[C>G]A,A[C>G]C,A[C>G]G,A[C>G]T,A[C>T]A,A[C>T]C,...,T[T>G]G,T[T>G]T,Gene_KO,Count,Protein_KO,Subpathway_KO,label,boot,Gene_KO_paired,Subpathway_KO_paired
0,0.026484,0.017352,0.006393,0.023744,0.007306,0.009132,0.007306,0.019178,0.017352,0.010046,...,0.001826,0.013699,EXO1,1095,Exonuclease 1,HR and HR regulation,2,True,EXO1_boot,HR and HR regulation_boot
1,0.047970,0.003690,0.011070,0.007380,0.011070,0.003690,0.033210,0.014760,0.014760,0.007380,...,0.011070,0.000000,RNF168,271,Ring finger protein 168,Checkpoint/DSB repair,2,True,RNF168_boot,Checkpoint/DSB repair_boot
2,0.044280,0.003690,0.011070,0.014760,0.007380,0.000000,0.029520,0.007380,0.029520,0.003690,...,0.000000,0.011070,RNF168,271,Ring finger protein 168,Checkpoint/DSB repair,2,True,RNF168_boot,Checkpoint/DSB repair_boot
3,0.023744,0.016438,0.007306,0.012785,0.009132,0.013699,0.007306,0.010046,0.021918,0.007306,...,0.000913,0.010959,EXO1,1095,Exonuclease 1,HR and HR regulation,2,True,EXO1_boot,HR and HR regulation_boot
4,0.047970,0.000000,0.007380,0.022140,0.007380,0.003690,0.022140,0.014760,0.033210,0.007380,...,0.000000,0.003690,RNF168,271,Ring finger protein 168,Checkpoint/DSB repair,2,True,RNF168_boot,Checkpoint/DSB repair_boot
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
MSH2_120_s2,0.011017,0.002119,0.000424,0.011017,0.002119,0.001271,0.000847,0.004237,0.066525,0.020763,...,0.000847,0.003814,MSH2,2360,MutS protein homolog 2,MMR,1,False,MSH2_real,MMR_real
MSH6_3_s8,0.006762,0.003005,0.001503,0.008640,0.000751,0.001127,0.000376,0.004132,0.060105,0.018032,...,0.001878,0.002254,MSH6,2662,MutS protein homolog 6,MMR,1,False,MSH6_real,MMR_real
MSH6_3_s4,0.003407,0.005679,0.000000,0.006246,0.001704,0.001136,0.000568,0.005679,0.056786,0.018739,...,0.000000,0.001136,MSH6,1761,MutS protein homolog 6,MMR,1,False,MSH6_real,MMR_real
PMS1_130_s2,0.036011,0.002770,0.000000,0.016620,0.000000,0.002770,0.000000,0.000000,0.047091,0.005540,...,0.000000,0.002770,PMS1,361,PMS1 protein homolog 1,MMR,1,False,PMS1_real,MMR_real


In [60]:
# ---------------------------------------------------------------------
# ✅ ADD ORIGINAL SPLIT AS SPLIT_10
# ---------------------------------------------------------------------
print("\nAdding original split (split_10) using real data...")

# --- Paths to your original real bootstrapped folds ---
path_trainfold = project_root / f"data/processed/bootstrapped_sameSplit/N_{N_train}"
path_testfold = project_root / f"data/processed/bootstrapped_sameSplit/N_{N_test}"

train_folds = [
    pd.read_pickle(path_trainfold / "train_1.pkl"),
    pd.read_pickle(path_trainfold / "train_2.pkl"),
    pd.read_pickle(path_trainfold / "train_3.pkl"),
]
df_test = pd.read_pickle(path_testfold / "test.pkl")

# Recreate labels for compatibility
df_test["label_name"] = df_test["Gene_KO"].map(gene_to_pathway)
df_test["label"] = df_test["label_name"].map(label_map)
df_test["Gene_KO_paired"] = df_test["Gene_KO"].astype(str)


# --- Get real (non-bootstrapped) samples for each fold ---
real_folds = []
for i, fold in enumerate(train_folds, start=1):
    df_real = fold[fold["boot"] == False].copy()

    # Recreate labels for compatibility
    df_real["label_name"] = df_real["Gene_KO"].map(gene_to_pathway)
    df_real["label"] = df_real["label_name"].map(label_map)
    df_real["Gene_KO_paired"] = df_real["Gene_KO"].astype(str)

    print(f"Fold {i}: {len(df_real)} real samples")
    real_folds.append(df_real)

# --- Combine real samples for train_all ---
df_train_all = pd.concat(real_folds, axis=0)
print(f"Train_all: {len(df_train_all)} real samples | Test: {len(df_test)}")


# --- Create split_10 directory ---
split_dir = out_root / "split_10"
split_dir.mkdir(parents=True, exist_ok=True)
subfolds_dir = split_dir / "sub_folds"
subfolds_dir.mkdir(exist_ok=True)

# --- Bootstrap each of the 3 original folds ---
XbootM_all, XbootD_all = [], []

for i, fold_df in enumerate(real_folds, start=1):
    fold_dir = subfolds_dir / f"fold_{i}"
    fold_dir.mkdir(exist_ok=True)

    print(f"→ Bootstrapping original fold {i} ({len(fold_df)} samples)")

    XbootM_train, XbootD_train = boot.bootstrap_dirichlet(
        fold_df, L=N_train, dirichlet_strength=ds, suffix=f"train_fold{i}", class_col="label"
    )

    save_bootstrapped_split(XbootM_train, f"train_fold{i}", fold_dir, prefix="M")
    save_bootstrapped_split(XbootD_train, f"train_fold{i}", fold_dir, prefix="D")

    XbootM_all.append(XbootM_train)
    XbootD_all.append(XbootD_train)

# --- Combine into train_all ---
XbootM_all = pd.concat(XbootM_all, axis=0)
XbootD_all = pd.concat(XbootD_all, axis=0)

save_bootstrapped_split(XbootM_all, "train_all", split_dir, prefix="M")
save_bootstrapped_split(XbootD_all, "train_all", split_dir, prefix="D")

# --- Bootstrap test set (same way as others) ---
print(f"→ Bootstrapping test (original, {len(df_test)} samples → 4000)...")
XbootM_test, XbootD_test = boot.bootstrap_dirichlet(
    df_test, L=N_test, dirichlet_strength=ds, suffix="test", class_col="label"
)

save_bootstrapped_split(XbootM_test, "test", split_dir, prefix="M")
save_bootstrapped_split(XbootD_test, "test", split_dir, prefix="D")

print("✅ Saved split_10 (original data)")



Adding original split (split_10) using real data...
Fold 1: 12 real samples
Fold 2: 10 real samples
Fold 3: 13 real samples
Train_all: 35 real samples | Test: 4000
→ Bootstrapping original fold 1 (12 samples)
→ Bootstrapping original fold 2 (10 samples)
→ Bootstrapping original fold 3 (13 samples)
→ Bootstrapping test (original, 4000 samples → 4000)...
✅ Saved split_10 (original data)


In [42]:
N_train = 100
path_trainfold = os.path.join(project_root, "data/processed/bootstrapped_sameSplit/N_{}".format(N_train)) 
N_test = 1000
path_testfold = os.path.join(project_root, "data/processed/bootstrapped_sameSplit/N_{}".format(N_test)) 

train_1 = pd.read_pickle(path_trainfold + "/train_1.pkl")
train_2 = pd.read_pickle(path_trainfold + "/train_2.pkl")
train_3 = pd.read_pickle(path_trainfold + "/train_3.pkl")
test = pd.read_pickle(path_testfold + "/test.pkl")

train_1 

,A[C>A]A,A[C>A]C,A[C>A]G,A[C>A]T,A[C>G]A,A[C>G]C,A[C>G]G,A[C>G]T,A[C>T]A,A[C>T]C,...,T[T>G]G,T[T>G]T,Gene_KO,Count,Protein_KO,Subpathway_KO,label,boot,Gene_KO_paired,Subpathway_KO_paired
0,0.026484,0.017352,0.006393,0.023744,0.007306,0.009132,0.007306,0.019178,0.017352,0.010046,...,0.001826,0.013699,EXO1,1095,Exonuclease 1,HR and HR regulation,2,True,EXO1_boot,HR and HR regulation_boot
1,0.047970,0.003690,0.011070,0.007380,0.011070,0.003690,0.033210,0.014760,0.014760,0.007380,...,0.011070,0.000000,RNF168,271,Ring finger protein 168,Checkpoint/DSB repair,2,True,RNF168_boot,Checkpoint/DSB repair_boot
2,0.044280,0.003690,0.011070,0.014760,0.007380,0.000000,0.029520,0.007380,0.029520,0.003690,...,0.000000,0.011070,RNF168,271,Ring finger protein 168,Checkpoint/DSB repair,2,True,RNF168_boot,Checkpoint/DSB repair_boot
3,0.023744,0.016438,0.007306,0.012785,0.009132,0.013699,0.007306,0.010046,0.021918,0.007306,...,0.000913,0.010959,EXO1,1095,Exonuclease 1,HR and HR regulation,2,True,EXO1_boot,HR and HR regulation_boot
4,0.047970,0.000000,0.007380,0.022140,0.007380,0.003690,0.022140,0.014760,0.033210,0.007380,...,0.000000,0.003690,RNF168,271,Ring finger protein 168,Checkpoint/DSB repair,2,True,RNF168_boot,Checkpoint/DSB repair_boot
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
MSH2_120_s2,0.011017,0.002119,0.000424,0.011017,0.002119,0.001271,0.000847,0.004237,0.066525,0.020763,...,0.000847,0.003814,MSH2,2360,MutS protein homolog 2,MMR,1,False,MSH2_real,MMR_real
MSH6_3_s8,0.006762,0.003005,0.001503,0.008640,0.000751,0.001127,0.000376,0.004132,0.060105,0.018032,...,0.001878,0.002254,MSH6,2662,MutS protein homolog 6,MMR,1,False,MSH6_real,MMR_real
MSH6_3_s4,0.003407,0.005679,0.000000,0.006246,0.001704,0.001136,0.000568,0.005679,0.056786,0.018739,...,0.000000,0.001136,MSH6,1761,MutS protein homolog 6,MMR,1,False,MSH6_real,MMR_real
PMS1_130_s2,0.036011,0.002770,0.000000,0.016620,0.000000,0.002770,0.000000,0.000000,0.047091,0.005540,...,0.000000,0.002770,PMS1,361,PMS1 protein homolog 1,MMR,1,False,PMS1_real,MMR_real


In [13]:
N_train = 100
path_trainfold = os.path.join(project_root, "data/processed/bootstrapped_sameSplit/N_{}".format(N_train)) 

train_1 = pd.read_pickle(path_trainfold + "/train_1.pkl")
train_2 = pd.read_pickle(path_trainfold + "/train_2.pkl")
train_3 = pd.read_pickle(path_trainfold + "/train_3.pkl")

print(train_1[train_1['boot'] == False].shape)
print(train_2[train_2['boot'] == False].shape)
print(train_3[train_3['boot'] == False].shape)


(12, 104)
(10, 104)
(13, 104)
